# What CAN'T Be Done on Snowpark Connect — Classic Spark Edition

This notebook demonstrates Spark features that **work on Classic Spark** but are **NOT supported on Snowpark Connect**.

Compare side-by-side with `04_unsupported_snowpark_connect.ipynb` to see the errors.

| # | Feature | Classic Spark | Snowpark Connect | Snowflake Alternative |
|---|---------|:---:|:---:|---|
| 1 | RDDs | Works | **FAILS** | Use DataFrame API |
| 2 | Structured Streaming | Works | **FAILS** | Snowpipe Streaming / Dynamic Tables |
| 3 | MLlib | Works | **FAILS** | Snowpark ML / Cortex ML |
| 4 | SparkContext (broadcast, accumulator) | Works | **FAILS** | Not needed — use DataFrames |
| 5 | Repartition / Coalesce | Works | No-op (silently ignored) | Snowflake auto-optimizes |

## Session Setup — Classic Spark

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Unsupported Features - Classic")
    .getOrCreate()
)

sc = spark.sparkContext
print(f"Spark version: {spark.version}")
print(f"SparkContext available: {sc is not None}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/10 21:22:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.3
SparkContext available: True


---
## 1. RDDs — Works on Classic, FAILS on Snowpark Connect
RDDs (Resilient Distributed Datasets) are Spark's low-level API. Snowpark Connect only supports the higher-level DataFrame API.

In [2]:
# Create an RDD from a Python list
rdd = sc.parallelize([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

# map — apply a function to each element
squared = rdd.map(lambda x: x ** 2)
print(f"Squared:  {squared.collect()}")

# filter — keep only elements matching a condition
evens = rdd.filter(lambda x: x % 2 == 0)
print(f"Evens:    {evens.collect()}")

# reduce — aggregate all elements
total = rdd.reduce(lambda a, b: a + b)
print(f"Sum:      {total}")

# flatMap — one-to-many transformation
expanded = rdd.flatMap(lambda x: [x, x * 10])
print(f"FlatMap:  {expanded.collect()}")

print("\n--- All RDD operations worked on Classic Spark ---")

Squared:  [1, 4, 9, 16, 25, 36, 49, 64, 81, 100]
Evens:    [2, 4, 6, 8, 10]
Sum:      55
FlatMap:  [1, 10, 2, 20, 3, 30, 4, 40, 5, 50, 6, 60, 7, 70, 8, 80, 9, 90, 10, 100]

--- All RDD operations worked on Classic Spark ---


---
## 2. Structured Streaming — Works on Classic, FAILS on Snowpark Connect
Spark Structured Streaming allows continuous processing of data streams. Snowpark Connect does not support `readStream` / `writeStream`.

> **Snowflake alternative:** Snowpipe Streaming, Dynamic Tables, or OpenFlow.

In [3]:
# Create a streaming DataFrame from a rate source (generates rows per second)
streaming_df = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 5)
    .load()
)

print(f"Is streaming: {streaming_df.isStreaming}")
print(f"Schema: {streaming_df.schema.simpleString()}")

# Start a streaming query — write to console for 3 seconds then stop
query = (
    streaming_df.writeStream
    .format("console")
    .outputMode("append")
    .start()
)

import time
time.sleep(3)
query.stop()

print("\n--- Structured Streaming worked on Classic Spark ---")

Is streaming: True
Schema: struct<timestamp:timestamp,value:bigint>


26/03/10 21:22:56 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /private/var/folders/8k/k5xk1xrn6p587fngwxrbg4c40000gn/T/temporary-d3d099de-ac31-45e2-bba5-fe7f556585f3. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/03/10 21:22:56 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+---------+-----+
|timestamp|value|
+---------+-----+
+---------+-----+

-------------------------------------------
Batch: 1
-------------------------------------------
+--------------------+-----+
|           timestamp|value|
+--------------------+-----+
|2026-03-10 21:22:...|    0|
|2026-03-10 21:22:...|    1|
|2026-03-10 21:22:...|    2|
|2026-03-10 21:22:...|    3|
|2026-03-10 21:22:...|    4|
+--------------------+-----+

-------------------------------------------
Batch: 2
-------------------------------------------
+--------------------+-----+
|           timestamp|value|
+--------------------+-----+
|2026-03-10 21:22:...|    5|
|2026-03-10 21:22:...|    6|
|2026-03-10 21:22:...|    7|
|2026-03-10 21:22:...|    8|
|2026-03-10 21:22:...|    9|
+--------------------+-----+


--- Structured Streaming worked on Classic Spark ---


---
## 3. MLlib — Works on Classic, FAILS on Snowpark Connect
Spark MLlib provides machine learning algorithms. Snowpark Connect does not support `pyspark.ml`.

> **Snowflake alternative:** Snowpark ML, Snowflake Model Registry, Cortex ML Functions.

In [4]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline

# Sample training data
training_data = spark.createDataFrame([
    (0, 1.0, 0.1, 0),
    (1, 0.2, 0.8, 1),
    (2, 0.9, 0.2, 0),
    (3, 0.1, 0.9, 1),
    (4, 0.8, 0.3, 0),
    (5, 0.3, 0.7, 1),
], ["id", "feature1", "feature2", "label"])

# Build a pipeline: assemble features -> train logistic regression
assembler = VectorAssembler(inputCols=["feature1", "feature2"], outputCol="features")
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=10)
pipeline = Pipeline(stages=[assembler, lr])

# Train
model = pipeline.fit(training_data)

# Predict
predictions = model.transform(training_data)
predictions.select("id", "feature1", "feature2", "label", "prediction").show()

print("--- MLlib pipeline worked on Classic Spark ---")

26/03/10 21:23:04 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/03/10 21:23:04 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS


+---+--------+--------+-----+----------+
| id|feature1|feature2|label|prediction|
+---+--------+--------+-----+----------+
|  0|     1.0|     0.1|    0|       0.0|
|  1|     0.2|     0.8|    1|       1.0|
|  2|     0.9|     0.2|    0|       0.0|
|  3|     0.1|     0.9|    1|       1.0|
|  4|     0.8|     0.3|    0|       0.0|
|  5|     0.3|     0.7|    1|       1.0|
+---+--------+--------+-----+----------+

--- MLlib pipeline worked on Classic Spark ---


---
## 4. SparkContext Direct Access — Works on Classic, FAILS on Snowpark Connect
SparkContext provides low-level primitives like broadcast variables and accumulators. These are not available in Snowpark Connect.

In [5]:
# Broadcast variable — share read-only data across all workers
lookup = {"A": "Apple", "B": "Banana", "C": "Cherry"}
broadcast_lookup = sc.broadcast(lookup)

rdd = sc.parallelize(["A", "B", "C", "A"])
result = rdd.map(lambda x: broadcast_lookup.value[x]).collect()
print(f"Broadcast lookup: {result}")

# Accumulator — a shared counter across workers
counter = sc.accumulator(0)
sc.parallelize([1, 2, 3, 4, 5]).foreach(lambda x: counter.add(1))
print(f"Accumulator count: {counter.value}")

# textFile — read a text file as RDD
import tempfile, os
tmp = os.path.join(tempfile.gettempdir(), "spark_demo.txt")
with open(tmp, "w") as f:
    f.write("hello world\nfoo bar\nhello spark")

lines = sc.textFile(tmp)
word_count = lines.flatMap(lambda l: l.split()).countByValue()
print(f"Word count: {dict(word_count)}")

print("\n--- SparkContext operations worked on Classic Spark ---")

Broadcast lookup: ['Apple', 'Banana', 'Cherry', 'Apple']
Accumulator count: 5
Word count: {'hello': 2, 'world': 1, 'foo': 1, 'bar': 1, 'spark': 1}

--- SparkContext operations worked on Classic Spark ---


---
## 5. Repartition / Coalesce — Works on Classic, No-op on Snowpark Connect
On Classic Spark, you control how data is partitioned. On Snowpark Connect, these calls are **silently ignored** — Snowflake handles partitioning automatically.

In [6]:
df = spark.createDataFrame([(i, f"row_{i}") for i in range(100)], ["id", "value"])

print(f"Default partitions: {df.rdd.getNumPartitions()}")

# Repartition to 8 partitions
df_8 = df.repartition(8)
print(f"After repartition(8): {df_8.rdd.getNumPartitions()}")

# Coalesce to 2 partitions
df_2 = df_8.coalesce(2)
print(f"After coalesce(2): {df_2.rdd.getNumPartitions()}")

print("\n--- Repartition/Coalesce worked on Classic Spark ---")
print("--- On Snowpark Connect, these are silently ignored (no-op) ---")

Default partitions: 10
After repartition(8): 8
After coalesce(2): 2

--- Repartition/Coalesce worked on Classic Spark ---
--- On Snowpark Connect, these are silently ignored (no-op) ---


---
## Summary

Everything above ran successfully on Classic Spark.

Now open **`04_unsupported_snowpark_connect.ipynb`** to see the same code **fail** on Snowpark Connect.

| Feature | Classic Spark | Snowpark Connect | Snowflake Alternative |
|---|:---:|:---:|---|
| RDDs (parallelize, map, reduce) | **Works** | Fails | DataFrame API |
| Structured Streaming (readStream) | **Works** | Fails | Snowpipe Streaming / Dynamic Tables |
| MLlib (Pipeline, LogisticRegression) | **Works** | Fails | Snowpark ML / Cortex ML |
| SparkContext (broadcast, accumulator) | **Works** | Fails | Not needed with DataFrames |
| Repartition / Coalesce | **Works** | No-op | Snowflake auto-optimizes |

In [7]:
spark.stop()
print("Session stopped.")

Session stopped.
